# Reconnaissance d'entités nommées

In [ ]:
# Cellule 1 — Installation
!pip install lxml pandas spacy
!python -m spacy download fr_core_news_md

In [ ]:
# Cellule 2 — Imports et chemins

from pathlib import Path
from lxml import etree
import pandas as pd
import spacy
import copy

TEI_IN = Path("Frêne_volume_1.xml")
TEI_OUT = Path("Frêne_volume_1_ner.xml")
ENTITIES_CSV = Path("Frêne_volume_1_entities.csv")

NS = {
    "tei": "http://www.tei-c.org/ns/1.0",
    "xml": "http://www.w3.org/XML/1998/namespace",
}

nlp = spacy.load("fr_core_news_md")

In [ ]:
# Cellule 3 — Extraction des lignes TEI

def extraire_lignes_tei(tei_path):
    """Extrait les lignes TEI avec leurs identifiants stables."""
    tree = etree.parse(str(tei_path))
    root = tree.getroot()

    rows = []

    for line in root.findall(".//tei:line", namespaces=NS):
        xml_id = line.get("{http://www.w3.org/XML/1998/namespace}id")
        n = line.get("n")
        text = "".join(line.itertext()).strip()

        if text:
            surface = line.getparent()
            while surface is not None and etree.QName(surface).localname != "surface":
                surface = surface.getparent()

            surface_id = None
            surface_n = None

            if surface is not None:
                surface_id = surface.get("{http://www.w3.org/XML/1998/namespace}id")
                surface_n = surface.get("n")

            rows.append({
                "xml_id": xml_id,
                "surface_id": surface_id,
                "surface_n": surface_n,
                "line_n": n,
                "text": text,
            })

    return pd.DataFrame(rows)


df_lignes = extraire_lignes_tei(TEI_IN)
df_lignes.head()

In [ ]:
# Cellule 4 — Reconnaissance d’entités nommées

def reconnaitre_entites(df):
    """Applique spaCy ligne par ligne et conserve les offsets."""
    rows = []

    for _, row in df.iterrows():
        doc = nlp(row["text"])

        for ent in doc.ents:
            rows.append({
                "xml_id": row["xml_id"],
                "surface_id": row["surface_id"],
                "surface_n": row["surface_n"],
                "line_n": row["line_n"],
                "entity_text": ent.text,
                "label": ent.label_,
                "start_char": ent.start_char,
                "end_char": ent.end_char,
            })

    return pd.DataFrame(rows)


df_entities = reconnaitre_entites(df_lignes)
df_entities.to_csv(ENTITIES_CSV, index=False, encoding="utf-8")
df_entities.head()

In [ ]:
# Cellule 5 — Mapping spaCy → TEI

SPACY_TO_TEI = {
    "PER": "persName",
    "PERSON": "persName",
    "LOC": "placeName",
    "GPE": "placeName",
    "ORG": "orgName",
    "MISC": "name",
    "DATE": "date",
}

In [ ]:
# Cellule 6 — Injection des entités dans les lignes TEI

def construire_ligne_annotee(text, entities):
    """Construit une ligne TEI annotée à partir du texte et des entités."""
    parent = etree.Element("tmp")
    pos = 0

    entities = sorted(entities, key=lambda e: e["start_char"])

    last_node = None

    for ent in entities:
        start = int(ent["start_char"])
        end = int(ent["end_char"])

        if start < pos:
            continue

        before = text[pos:start]
        ent_text = text[start:end]

        tei_tag = SPACY_TO_TEI.get(ent["label"], "name")

        if last_node is None:
            parent.text = (parent.text or "") + before
        else:
            last_node.tail = (last_node.tail or "") + before

        element = etree.Element(f"{{http://www.tei-c.org/ns/1.0}}{tei_tag}")
        element.text = ent_text
        element.set("type", ent["label"])

        parent.append(element)
        last_node = element

        pos = end

    after = text[pos:]

    if last_node is None:
        parent.text = (parent.text or "") + after
    else:
        last_node.tail = (last_node.tail or "") + after

    return parent


def injecter_entites_dans_tei(tei_in, entities_df, tei_out):
    """Réinjecte les entités nommées dans une copie du TEI."""
    tree = etree.parse(str(tei_in))
    root = tree.getroot()

    entities_by_line = {
        xml_id: group.to_dict("records")
        for xml_id, group in entities_df.groupby("xml_id")
    }

    for line in root.findall(".//tei:line", namespaces=NS):
        xml_id = line.get("{http://www.w3.org/XML/1998/namespace}id")

        if xml_id not in entities_by_line:
            continue

        original_text = "".join(line.itertext())
        annotated = construire_ligne_annotee(
            original_text,
            entities_by_line[xml_id],
        )

        line.clear(keep_tail=True)
        line.set("{http://www.w3.org/XML/1998/namespace}id", xml_id)

        line.text = annotated.text

        for child in annotated:
            line.append(child)

    tree.write(
        str(tei_out),
        encoding="UTF-8",
        xml_declaration=True,
        pretty_print=True,
    )

    print(f"TEI enrichi créé : {tei_out}")


injecter_entites_dans_tei(TEI_IN, df_entities, TEI_OUT)

In [ ]:
# Cellule 7 — Vérification rapide

tree = etree.parse(str(TEI_OUT))

print("persName :", len(tree.findall(".//tei:persName", namespaces=NS)))
print("placeName :", len(tree.findall(".//tei:placeName", namespaces=NS)))
print("orgName :", len(tree.findall(".//tei:orgName", namespaces=NS)))
print("date :", len(tree.findall(".//tei:date", namespaces=NS)))